# Listener Prior — Retrieval Model + Shared Index (3 datasets, Option A)

Trains a **retrieval** model (SentenceTransformers) to predict **keyterms** and **keywords** for the **next USER utterance** only (USER-turn prediction for Deepgram STT injection).
Selects the **best checkpoint by Recall@20**, **separately for keyterms and keywords**, and saves:

- `best_model/` (encoder)
- `shared_index/` (FAISS index + candidates + metadata)
- `best_metrics.json`, `performance.json`

Also prints example conversations with predictions.

## Datasets
- Public dialog datasets (repo loader or HF fallback)
- `examples/multi_restaurant_phone_orders_2000.csv`


In [ ]:
# ---------------------------
# CONFIG (edit as needed)
# ---------------------------
REPO_URL = "https://github.com/ebilal/fSTT.git"
PROJECT_DIR = "/content/listener-prior"

BASE_EMBEDDER = "sentence-transformers/all-MiniLM-L6-v2"  # all-MiniLM-L6-v2, paraphrase-MiniLM-L3-v2
HISTORY_TURNS = 8
MAX_KEYWORDS = 30
MAX_KEYTERMS = 30

# Training
EPOCHS = 6
BATCH_SIZE = 256          # effective batch size
GRAD_ACCUM_STEPS = 4      # micro-batch = BATCH_SIZE // GRAD_ACCUM_STEPS (=64)
LR = 2e-5
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPS = 1e-8

# Selection metric
TOPK = 20
# We select best by KEYTERMS Recall@20 (and still log keywords Recall@20)
SELECTION_FIELD = "val_recall@20_keyterms"
VAL_EXAMPLES_FOR_FAST_EVAL = None  # e.g. 5000

# Saving (Google Drive) — each run gets a unique timestamped folder
from datetime import datetime
RUN_TAG = "retrieval_minilm_l3_user_only"
RUN_NAME = f"{RUN_TAG}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
DRIVE_ROOT = "/content/drive/MyDrive/listener_prior_runs"
RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
BEST_DIR = f"{RUN_DIR}/best_model"
INDEX_DIR = f"{RUN_DIR}/shared_index"
print(f"Run artifacts will be saved to: {RUN_DIR}")

RESTAURANT_CSV = "examples/multi_restaurant_phone_orders_5000.csv"


## Mount Google Drive + set HF_TOKEN from Colab Secrets


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

for k in ['HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN', 'HUGGING_FACE_HUB_TOKEN']:
    os.environ.pop(k, None)

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token set from Colab Secrets.')
else:
    print('No HF_TOKEN secret found. Public datasets/models should still work without it.')


## Clone repo + install dependencies


In [ ]:
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"

!python -m pip install -U pip
!grep -v '^torch' requirements.txt > /tmp/requirements_no_torch.txt
!python -m pip install -r /tmp/requirements_no_torch.txt
!python -m pip install faiss-cpu accelerate

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())


## Load conversations (public + restaurant CSV)


In [ ]:
import os, json
import pandas as pd
from typing import List, Dict, Any

def _normalize_role(role: str) -> str:
    role = (role or "").strip().lower()
    if role in {"user", "customer", "human", "client", "guest"}:
        return "USER"
    if role in {"system", "assistant", "agent", "bot", "server"}:
        return "SYSTEM"
    return "SYSTEM"

def load_public_conversations() -> List[Dict[str, Any]]:
    convos: List[Dict[str, Any]] = []
    try:
        from src import data as data_mod
        for fn_name in ["load_dual_dataset_conversations", "load_public_conversations", "get_dual_dataset"]:
            if hasattr(data_mod, fn_name):
                out = getattr(data_mod, fn_name)()
                if isinstance(out, dict):
                    for _, v in out.items():
                        if isinstance(v, list):
                            convos.extend(v)
                elif isinstance(out, list):
                    convos = out
                if convos:
                    print(f"Loaded public via src.data.{fn_name}: {len(convos)}")
                    return convos
    except Exception as e:
        print("Repo loader not available / failed:", repr(e))

    import datasets

    def _extract_text(turn: Dict) -> str:
        for key in ["text", "utterance", "transcript", "content", "sentence"]:
            if key in turn and isinstance(turn[key], str):
                return turn[key]
        return ""

    def _extract_turns_from_item(item: Dict) -> List[Dict[str, str]]:
        turns: List[Dict[str, str]] = []

        # DailyDialog (roskoN/dailydialog) style: a single list of utterances.
        # Roles are not explicitly provided; alternate USER/SYSTEM by index.
        if "utterances" in item and isinstance(item["utterances"], list) and item["utterances"]:
            utterances = item["utterances"]
            for i, text in enumerate(utterances):
                if not isinstance(text, str):
                    continue
                text = text.strip()
                if not text:
                    continue
                role = "USER" if i % 2 == 0 else "SYSTEM"
                turns.append({"speaker": role, "text": text})
            if turns:
                return turns

        if "turns" in item and isinstance(item["turns"], dict):
            speakers = item["turns"].get("speaker") or []
            utterances = item["turns"].get("utterance") or []
            for speaker, text in zip(speakers, utterances):
                role = "USER" if str(speaker) == "0" else "SYSTEM"
                text = (text or "").strip()
                if text:
                    turns.append({"speaker": role, "text": text})
            return turns

        turns_list = None
        for key in ["turns", "dialogue", "dialog", "utterances", "messages"]:
            if key in item and isinstance(item[key], list):
                turns_list = item[key]
                break
        if turns_list is None:
            return []

        for idx, turn in enumerate(turns_list):
            if isinstance(turn, str):
                role = "USER" if idx % 2 == 0 else "SYSTEM"
                text = turn
            elif isinstance(turn, dict):
                role = _normalize_role(turn.get("speaker") or turn.get("role") or turn.get("participant") or "")
                text = _extract_text(turn)
                if not text and "utterances" in turn and isinstance(turn["utterances"], str):
                    text = turn["utterances"]
            else:
                continue
            text = (text or "").strip()
            if text:
                turns.append({"speaker": role, "text": text})
        return turns

    # Prefer the HF-hosted DailyDialog mirror (no external zip link), with legacy fallback.
    DATASET_SOURCES = [
        ("dailydialog", ["roskoN/dailydialog", "daily_dialog"]),
        ("multiwoz", ["pfb30/multi_woz_v22"]),
    ]

    token = (
        os.environ.get("HF_TOKEN")
        or os.environ.get("HUGGINGFACE_HUB_TOKEN")
        or os.environ.get("HUGGING_FACE_HUB_TOKEN")
        or None
    )

    def _load(repo_id: str):
        if token:
            try:
                return datasets.load_dataset(repo_id, split="train", token=token, trust_remote_code=True)
            except TypeError:
                return datasets.load_dataset(repo_id, split="train", use_auth_token=token, trust_remote_code=True)
        return datasets.load_dataset(repo_id, split="train", trust_remote_code=True)

    print("Using HF fallback datasets:", DATASET_SOURCES)

    for ds_name, repo_ids in DATASET_SOURCES:
        dataset = None
        used_repo = None
        for repo_id in repo_ids:
            try:
                dataset = _load(repo_id)
                used_repo = repo_id
                break
            except Exception as e:
                print(f"Skipping {repo_id} due to load error: {repr(e)}")
                continue
        if dataset is None:
            continue
        if ds_name == "dailydialog" and used_repo == "daily_dialog":
            print("Warning: using legacy 'daily_dialog' source; prefer 'roskoN/dailydialog'.")

        for i, item in enumerate(dataset):
            turns = _extract_turns_from_item(item)
            if not turns:
                dialog = item.get("dialog")
                if isinstance(dialog, list):
                    turns = [{"speaker": "USER" if j % 2 == 0 else "SYSTEM", "text": str(t)} for j, t in enumerate(dialog) if isinstance(t, str)]
            if len(turns) >= 2:
                convos.append({"dialog_id": f"{used_repo}:train:{i}", "turns": turns})

    print("Loaded HF public conversations:", len(convos))
    return convos

def load_restaurant_csv(path: str) -> List[Dict[str, Any]]:
    assert os.path.exists(path), f"Missing CSV at {path}."
    df = pd.read_csv(path)
    required = {"dialog_id", "utterance_id", "speaker", "text"}
    assert required.issubset(set(df.columns)), f"CSV must include {required}, got {set(df.columns)}"
    convos: List[Dict[str, Any]] = []
    for did, g in df.sort_values(["dialog_id", "utterance_id"]).groupby("dialog_id"):
        turns = [
            {"speaker": _normalize_role(str(r["speaker"])), "text": str(r["text"])}
            for _, r in g.iterrows()
        ]
        if len(turns) >= 2:
            convos.append({"dialog_id": f"restaurant:{did}", "turns": turns})
    print("Loaded restaurant conversations:", len(convos))
    return convos

public_convos = load_public_conversations()
restaurant_convos = load_restaurant_csv(RESTAURANT_CSV)
all_convos = public_convos + restaurant_convos
print("TOTAL conversations:", len(all_convos))


In [ ]:
# Sanity check: show a few dialogs from each source
from itertools import islice

def _print_samples(convos, label, n=2):
    print("\n" + "="*80)
    print(f"{label} (showing {n})")
    print("="*80)
    for c in list(islice(convos, n)):
        print("dialog_id:", c["dialog_id"])
        for t in c["turns"][:3]:
            print(f"  {t['speaker']}: {t['text']}")
        if len(c["turns"]) > 3:
            print("  ...")

public_only = [c for c in all_convos if not c["dialog_id"].startswith("restaurant:")]
restaurant_only = [c for c in all_convos if c["dialog_id"].startswith("restaurant:")]

_print_samples(public_only, "Public datasets")
_print_samples(restaurant_only, "Restaurant CSV")


## Build Option A examples + global candidate pool


In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

def build_examples(convos, history_turns: int):
    # --- Pass 1: collect all USER-turn targets + history strings ---
    raw = []
    for c in convos:
        turns = c["turns"]
        for t in range(1, len(turns)):
            target = turns[t]
            # Only predict USER utterances (the ones we need Deepgram hints for)
            if target["speaker"] != "USER":
                continue
            hist = turns[max(0, t - history_turns):t]
            inp = "\n".join([f'{h["speaker"]}: {h["text"]}' for h in hist]).strip()
            raw.append({
                "dialog_id": c["dialog_id"],
                "turn_idx": t,
                "input_text": inp,
                "target_text_raw": target["text"],
            })
    if not raw:
        return []

    all_texts = [r["target_text_raw"] for r in raw]
    print(f"Collected {len(all_texts)} USER-turn targets. Running batched TF-IDF …")

    # --- Batch TF-IDF: one vectorizer per ngram range, over the whole corpus ---
    def batch_top_terms(texts, ngram_range, max_items):
        try:
            vec = TfidfVectorizer(stop_words="english", ngram_range=ngram_range, min_df=1)
            tfidf = vec.fit_transform(texts).tocsr()
        except ValueError:
            return [[] for _ in texts]
        if tfidf.shape[1] == 0:
            return [[] for _ in texts]
        names = vec.get_feature_names_out()
        results = []
        for i in range(tfidf.shape[0]):
            row = tfidf[i]
            if row.nnz == 0:
                results.append([])
                continue
            cols = row.indices
            top = np.argsort(-row.data)[:max_items]
            results.append([names[cols[j]] for j in top])
        return results

    all_keywords = batch_top_terms(all_texts, (1, 1), MAX_KEYWORDS)
    print("  keywords (unigrams) done")
    all_keyterms = batch_top_terms(all_texts, (2, 3), MAX_KEYTERMS)
    print("  keyterms (2-3 grams) done")

    # --- Regex patterns for numbers / times / dates (matches src/prior.py) ---
    _NUMERIC_RE = re.compile(r"\b\d{1,4}(?::\d{2})?\b")
    _TIME_RE    = re.compile(r"\b\d{1,2}(?:am|pm)\b", re.IGNORECASE)
    _DATE_RE    = re.compile(r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\w*\b", re.IGNORECASE)

    def _dedupe(items):
        seen = set(); out = []
        for x in items:
            k = x.lower()
            if k not in seen:
                seen.add(k); out.append(x)
        return out

    # --- Pass 2: assemble examples with pre-computed terms ---
    exs = []
    for i, r in enumerate(raw):
        txt = r["target_text_raw"]
        numeric = _NUMERIC_RE.findall(txt) + _TIME_RE.findall(txt) + _DATE_RE.findall(txt)
        keywords = _dedupe(all_keywords[i] + numeric)[:MAX_KEYWORDS]
        keyterms = _dedupe(all_keyterms[i])[:MAX_KEYTERMS]

        tgt = "keyterms: " + "; ".join(keyterms) + "\n"
        tgt += "keywords: " + "; ".join(keywords)

        exs.append({
            "dialog_id": r["dialog_id"],
            "turn_idx": r["turn_idx"],
            "input_text": r["input_text"],
            "target_text": tgt,
        })
    return exs

examples = build_examples(all_convos, HISTORY_TURNS)
candidates = sorted(set(e["target_text"] for e in examples if e["target_text"]))
print("examples:", len(examples), "unique candidates:", len(candidates))


## Split train/val/test by dialog id


In [ ]:
from collections import defaultdict
import random

rng = random.Random(7)
by_dialog = defaultdict(list)
for e in examples:
    by_dialog[e["dialog_id"]].append(e)

dialog_ids = list(by_dialog.keys())
rng.shuffle(dialog_ids)

n = len(dialog_ids)
n_train = int(0.8*n)
n_val = int(0.1*n)

train_ids = set(dialog_ids[:n_train])
val_ids   = set(dialog_ids[n_train:n_train+n_val])
test_ids  = set(dialog_ids[n_train+n_val:])

train_ex = [e for did in train_ids for e in by_dialog[did]]
val_ex   = [e for did in val_ids for e in by_dialog[did]]
test_ex  = [e for did in test_ids for e in by_dialog[did]]

print("dialogs:", n, "train/val/test:", len(train_ids), len(val_ids), len(test_ids))
print("examples:", len(train_ex), len(val_ex), len(test_ex))


## Recall@20 metrics split: keyterms vs keywords


In [ ]:
def parse_keyterms_keywords(text: str):
    keyterms = []
    keywords = []
    for line in text.splitlines():
        line = line.strip()
        if line.lower().startswith("keyterms:"):
            rhs = line.split(":",1)[1]
            keyterms = [t.strip() for t in rhs.split(";") if t.strip()]
        elif line.lower().startswith("keywords:"):
            rhs = line.split(":",1)[1]
            keywords = [t.strip() for t in rhs.split(";") if t.strip()]
    # de-dup preserving order
    def dedup(xs):
        seen=set(); out=[]
        for x in xs:
            if x not in seen:
                seen.add(x); out.append(x)
        return out
    return dedup(keyterms), dedup(keywords)

def recall_at_k(gt_list, pred_list, k):
    gt_set = set(gt_list)
    if not gt_set:
        return 0.0
    topk = pred_list[:k]
    return len(set(topk) & gt_set) / len(gt_set)

def recall20_from_retrieved(gt_text: str, retrieved_texts: list[str], k=20):
    gt_terms, gt_words = parse_keyterms_keywords(gt_text)

    pred_terms=[]
    pred_words=[]
    for cand in retrieved_texts:
        ct, cw = parse_keyterms_keywords(cand)
        pred_terms.extend(ct)
        pred_words.extend(cw)

    # unique preserving order
    def uniq(xs):
        seen=set(); out=[]
        for x in xs:
            if x not in seen:
                seen.add(x); out.append(x)
        return out

    pred_terms = uniq(pred_terms)
    pred_words = uniq(pred_words)

    return {
        "recall@20_keyterms": recall_at_k(gt_terms, pred_terms, k),
        "recall@20_keywords": recall_at_k(gt_words, pred_words, k),
    }


## Train retrieval model + select best by VAL Recall@20 (keyterms)


In [ ]:
import os, json, math
import numpy as np
import faiss
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sentence_transformers import SentenceTransformer, InputExample, losses
from tqdm.auto import tqdm

os.makedirs(RUN_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(BASE_EMBEDDER, device=device)
loss_fn = losses.MultipleNegativesRankingLoss(model)

# Micro-batch fits in GPU; effective batch = micro_batch * GRAD_ACCUM_STEPS
micro_batch = BATCH_SIZE // GRAD_ACCUM_STEPS
print(f"Effective batch: {BATCH_SIZE} = {micro_batch} micro-batch × {GRAD_ACCUM_STEPS} accum steps")

train_pairs = [InputExample(texts=[e["input_text"], e["target_text"]]) for e in train_ex]
train_dl = DataLoader(
    train_pairs,
    batch_size=micro_batch,
    shuffle=True,
    drop_last=True,
    collate_fn=model.smart_batching_collate,
)

# Optimizer + linear warmup scheduler
optimizer = AdamW(
    model.parameters(),
    lr=LR, weight_decay=WEIGHT_DECAY,
    betas=(ADAM_BETA1, ADAM_BETA2), eps=ADAM_EPS,
)
steps_per_epoch = math.ceil(len(train_dl) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

from torch.optim.lr_scheduler import LambdaLR
def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / max(1.0, float(warmup_steps))
    return max(0.0, float(total_steps - current_step) / max(1.0, float(total_steps - warmup_steps)))
scheduler = LambdaLR(optimizer, lr_lambda)

print(f"steps/epoch: {steps_per_epoch}, total: {total_steps}, warmup: {warmup_steps}")

# ---------- helpers ----------
def build_faiss_index(m, cand_texts):
    emb = m.encode(cand_texts, batch_size=256, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    dim = emb.shape[1]
    idx = faiss.IndexFlatIP(dim)
    idx.add(emb.astype(np.float32))
    return idx

@torch.no_grad()
def eval_metrics(m, cand_texts, eval_ex, limit=None):
    if limit is not None and limit < len(eval_ex):
        eval_ex = eval_ex[:limit]
    idx = build_faiss_index(m, cand_texts)
    bs = 512
    sum_terms=0.0; sum_words=0.0
    for i in range(0, len(eval_ex), bs):
        chunk = eval_ex[i:i+bs]
        queries = [e["input_text"] for e in chunk]
        q_emb = m.encode(queries, batch_size=256, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        D, I = idx.search(q_emb.astype(np.float32), TOPK)
        for row, ex in enumerate(chunk):
            retrieved = [cand_texts[j] for j in I[row] if j >= 0]
            m20 = recall20_from_retrieved(ex["target_text"], retrieved, k=TOPK)
            sum_terms += m20["recall@20_keyterms"]
            sum_words += m20["recall@20_keywords"]
    n = max(1, len(eval_ex))
    return {"recall@20_keyterms": float(sum_terms/n), "recall@20_keywords": float(sum_words/n)}

def save_best(m, val_m, epoch):
    os.makedirs(BEST_DIR, exist_ok=True)
    m.save(BEST_DIR)
    with open(os.path.join(RUN_DIR, "best_metrics.json"), "w") as f:
        json.dump({"selection_metric": SELECTION_FIELD, "epoch": epoch, **val_m, "history": history}, f, indent=2)
    print("  ✓ Saved BEST model to:", BEST_DIR)
    # Save shared FAISS index alongside the best model
    os.makedirs(INDEX_DIR, exist_ok=True)
    cand_emb = m.encode(candidates, batch_size=256, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    best_idx = faiss.IndexFlatIP(cand_emb.shape[1])
    best_idx.add(cand_emb.astype(np.float32))
    faiss.write_index(best_idx, os.path.join(INDEX_DIR, "index.faiss"))
    with open(os.path.join(INDEX_DIR, "candidates.json"), "w") as f:
        json.dump(candidates, f)
    with open(os.path.join(INDEX_DIR, "meta.json"), "w") as f:
        json.dump({
            "encoder_dir": BEST_DIR, "encoder_base": BASE_EMBEDDER,
            "normalized_embeddings": True, "topk": TOPK,
            "candidate_count": len(candidates), "epoch": epoch, **val_m,
        }, f, indent=2)
    print("  ✓ Saved shared index to:", INDEX_DIR)

# ---------- training loop with gradient accumulation ----------
best = {SELECTION_FIELD: -1.0, "epoch": None}
history = []
global_step = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\n===== EPOCH {epoch}/{EPOCHS} =====")
    model.train()
    optimizer.zero_grad()
    epoch_loss = 0.0
    pbar = tqdm(enumerate(train_dl), total=len(train_dl), desc=f"Epoch {epoch}")

    for micro_step, batch in pbar:
        features, labels = batch
        features = [
            {k: v.to(model.device) for k, v in f.items()} for f in features
        ]
        step_loss = loss_fn(features, labels.to(model.device))
        step_loss = step_loss / GRAD_ACCUM_STEPS
        step_loss.backward()
        epoch_loss += step_loss.item()

        if (micro_step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            pbar.set_postfix(loss=f"{step_loss.item() * GRAD_ACCUM_STEPS:.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    # Handle leftover micro-batches at end of epoch
    if len(train_dl) % GRAD_ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        global_step += 1

    avg_loss = epoch_loss / max(1, len(train_dl))
    print(f"  avg loss: {avg_loss:.5f}")

    # --- Validation ---
    model.eval()
    val_m = eval_metrics(model, candidates, val_ex, limit=VAL_EXAMPLES_FOR_FAST_EVAL)
    print(f"  VAL Recall@20 keyterms: {val_m['recall@20_keyterms']:.6f} | keywords: {val_m['recall@20_keywords']:.6f}")
    row = {"epoch": epoch, "avg_loss": avg_loss,
           "val_recall@20_keyterms": val_m["recall@20_keyterms"],
           "val_recall@20_keywords": val_m["recall@20_keywords"]}
    history.append(row)

    if val_m["recall@20_keyterms"] > best[SELECTION_FIELD]:
        best = {SELECTION_FIELD: val_m["recall@20_keyterms"], "epoch": epoch, **val_m}
        save_best(model, val_m, epoch)

print("\nBEST:", best)


## Test evaluation (best) + save performance.json


In [ ]:
import os, json
from sentence_transformers import SentenceTransformer
import torch

best_model = SentenceTransformer(BEST_DIR, device=("cuda" if torch.cuda.is_available() else "cpu"))
test_m = eval_metrics(best_model, candidates, test_ex, limit=None)
print(f"TEST Recall@20 keyterms: {test_m['recall@20_keyterms']:.6f} | keywords: {test_m['recall@20_keywords']:.6f}")

perf = {
    "run_name": RUN_NAME,
    "base_embedder": BASE_EMBEDDER,
    "history_turns": HISTORY_TURNS,
    "max_keywords": MAX_KEYWORDS,
    "max_keyterms": MAX_KEYTERMS,
    "selection_metric": SELECTION_FIELD,
    "best_epoch": best.get("epoch"),
    "best_val_recall@20_keyterms": best.get("recall@20_keyterms"),
    "best_val_recall@20_keywords": best.get("recall@20_keywords"),
    "test_recall@20_keyterms": test_m["recall@20_keyterms"],
    "test_recall@20_keywords": test_m["recall@20_keywords"],
    "num_candidates": len(candidates),
    "num_train_examples": len(train_ex),
    "num_val_examples": len(val_ex),
    "num_test_examples": len(test_ex),
}
with open(os.path.join(RUN_DIR, "performance.json"), "w") as f:
    json.dump(perf, f, indent=2)
print("Saved:", os.path.join(RUN_DIR, "performance.json"))


## Update shared index metadata with test metrics


In [ ]:
# Update shared index meta.json with final test metrics
# (index.faiss + candidates.json were already saved with the best checkpoint)
import os, json

meta_path = os.path.join(INDEX_DIR, "meta.json")
meta = json.load(open(meta_path))
meta["test_recall@20_keyterms"] = perf["test_recall@20_keyterms"]
meta["test_recall@20_keywords"] = perf["test_recall@20_keywords"]
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Updated index meta with test metrics:", meta_path)


## Example predictions (loaded from saved Drive artifacts)

Loads the best model + FAISS index entirely from Google Drive (no in-memory
variables) to verify the saved artifacts work end-to-end.


In [ ]:
import os, json, faiss, numpy as np, torch
from sentence_transformers import SentenceTransformer

# ---------- Load everything from saved Drive artifacts ----------
print(f"Loading encoder from: {BEST_DIR}")
saved_model = SentenceTransformer(BEST_DIR, device="cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading FAISS index from: {INDEX_DIR}")
saved_idx = faiss.read_index(os.path.join(INDEX_DIR, "index.faiss"))
with open(os.path.join(INDEX_DIR, "candidates.json")) as f:
    saved_cands = json.load(f)
with open(os.path.join(INDEX_DIR, "meta.json")) as f:
    saved_meta = json.load(f)

print(f"  candidates: {len(saved_cands)}, index vectors: {saved_idx.ntotal}")
print(f"  meta: {json.dumps(saved_meta, indent=2)}")

# ---------- Prediction helper (uses only saved artifacts) ----------
def predict_from_saved(history_text: str, topk=20):
    """Encode history, search saved index, aggregate keyterms/keywords."""
    q = saved_model.encode([history_text], convert_to_numpy=True,
                           normalize_embeddings=True).astype(np.float32)
    D, I = saved_idx.search(q, topk)
    retrieved = [saved_cands[j] for j in I[0] if j >= 0]

    pred_terms, pred_words = [], []
    for cand in retrieved:
        ct, cw = parse_keyterms_keywords(cand)
        pred_terms.extend(ct)
        pred_words.extend(cw)

    # de-dup preserving order
    def uniq(xs):
        seen = set(); out = []
        for x in xs:
            if x not in seen:
                seen.add(x); out.append(x)
        return out

    return uniq(pred_terms)[:topk], uniq(pred_words)[:topk], retrieved

# ---------- Run on a few test examples ----------
for ex in test_ex[:3]:
    pt, pw, retrieved = predict_from_saved(ex["input_text"], topk=TOPK)
    gt_t, gt_w = parse_keyterms_keywords(ex["target_text"])
    ov_t = sorted(set(pt) & set(gt_t))
    ov_w = sorted(set(pw) & set(gt_w))
    print("=" * 90)
    print("HISTORY:\n", ex["input_text"][:800])
    print("\nGT keyterms:", gt_t[:30])
    print("PRED keyterms@20:", pt)
    print("Overlap keyterms:", ov_t,
          f"(recall={len(ov_t)}/{len(set(gt_t)) if gt_t else 0})")
    print("\nGT keywords:", gt_w[:30])
    print("PRED keywords@20:", pw)
    print("Overlap keywords:", ov_w,
          f"(recall={len(ov_w)}/{len(set(gt_w)) if gt_w else 0})")
    print()


## Deepgram injection test (MP3 file)

Upload an MP3 of a recorded conversation and provide the conversation history
that preceded the audio. The cell will:

1. Predict keyterms/keywords from the history using the saved model + index
2. Transcribe the MP3 **without** hints (baseline)
3. Transcribe the MP3 **with** the predicted hints injected
4. Print both transcripts side-by-side so you can compare

Requires `DG_API_KEY` in Colab Secrets and `pip install deepgram-sdk`.


In [ ]:
!pip install -q deepgram-sdk

import os
from google.colab import userdata, files

# --- Config -----------------------------------------------------------
# Upload an MP3 (or wav/ogg/etc.) of the user's NEXT utterance to test.
# Provide the conversation history that happened BEFORE this audio.
AUDIO_PATH = None          # set below via upload, or hardcode a Drive path
DG_MODEL   = "nova-3"      # "nova-3" uses keyterm; "nova-2" uses keywords

EXAMPLE_HISTORY = """\
SYSTEM: Hi, thanks for calling. What can I get for you?
USER: I'd like to place an order for pickup.
SYSTEM: Sure, go ahead."""
# ----------------------------------------------------------------------

# Get Deepgram API key from Colab Secrets
try:
    DG_API_KEY = userdata.get("DG_API_KEY")
except Exception:
    DG_API_KEY = os.environ.get("DG_API_KEY")
assert DG_API_KEY, "Set DG_API_KEY in Colab Secrets (or env var)."

# Upload audio file if AUDIO_PATH is not set
if not AUDIO_PATH:
    print("Upload an audio file (mp3/wav/ogg/flac):")
    uploaded = files.upload()
    AUDIO_PATH = list(uploaded.keys())[0]
    print(f"Using: {AUDIO_PATH}")

# --- Step 1: predict keyterms/keywords from history -------------------
pred_keyterms, pred_keywords, _ = predict_from_saved(EXAMPLE_HISTORY, topk=TOPK)

# Merge + de-dupe into a single hint list
hints = []
seen = set()
for term in pred_keyterms + pred_keywords:
    t = term.strip()
    if t and t.lower() not in seen:
        seen.add(t.lower())
        hints.append(t)
hints = hints[:100]  # Deepgram limit: 500 tokens; 100 terms is safe

print(f"\nPredicted keyterms ({len(pred_keyterms)}):", pred_keyterms[:15])
print(f"Predicted keywords ({len(pred_keywords)}):", pred_keywords[:15])
print(f"Hints for Deepgram ({len(hints)}):", hints[:20])

# --- Step 2: transcribe WITHOUT hints (baseline) ---------------------
from deepgram import DeepgramClient, PrerecordedOptions
import httpx

dg = DeepgramClient(DG_API_KEY)

with open(AUDIO_PATH, "rb") as audio:
    source = {"buffer": audio, "mimetype": f"audio/{AUDIO_PATH.rsplit('.', 1)[-1]}"}
    baseline_opts = PrerecordedOptions(
        model=DG_MODEL,
        smart_format=True,
    )
    baseline_resp = dg.listen.rest.v("1").transcribe_file(
        source, baseline_opts, timeout=httpx.Timeout(300.0, connect=10.0)
    )

baseline_text = baseline_resp.results.channels[0].alternatives[0].transcript
print("\n" + "=" * 80)
print("BASELINE (no hints):")
print(baseline_text)

# --- Step 3: transcribe WITH predicted hints --------------------------
with open(AUDIO_PATH, "rb") as audio:
    source = {"buffer": audio, "mimetype": f"audio/{AUDIO_PATH.rsplit('.', 1)[-1]}"}

    if "nova-3" in DG_MODEL or "flux" in DG_MODEL:
        # Nova-3 / Flux: use keyterm parameter
        boosted_opts = PrerecordedOptions(
            model=DG_MODEL,
            smart_format=True,
            keyterm=hints,
        )
    else:
        # Nova-2 and older: use keywords parameter
        boosted_opts = PrerecordedOptions(
            model=DG_MODEL,
            smart_format=True,
            keywords=hints,
        )

    boosted_resp = dg.listen.rest.v("1").transcribe_file(
        source, boosted_opts, timeout=httpx.Timeout(300.0, connect=10.0)
    )

boosted_text = boosted_resp.results.channels[0].alternatives[0].transcript
print("\n" + "=" * 80)
print(f"WITH HINTS ({DG_MODEL}, {len(hints)} terms):")
print(boosted_text)

# --- Step 4: compare --------------------------------------------------
print("\n" + "=" * 80)
print("COMPARISON:")
if baseline_text == boosted_text:
    print("  Transcripts are identical (hints had no effect on this audio).")
else:
    # Simple word-level diff highlighting
    base_words = baseline_text.split()
    boost_words = boosted_text.split()
    diffs = [(b, h) for b, h in zip(base_words, boost_words) if b != h]
    if diffs:
        print("  Word-level differences (baseline → boosted):")
        for bw, hw in diffs[:20]:
            print(f"    {bw!r} → {hw!r}")
    if len(boost_words) != len(base_words):
        print(f"  Length changed: {len(base_words)} → {len(boost_words)} words")
